In [1]:
import pandas as pd
import sqlite3

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')

# We need to rebuild our dataframes here since this is a separate notebook
import requests

countries = "JAM;TTO;BRB;BHS;DOM;HTI;GUY;BLZ;GRD"
params = {"format": "json", "per_page": 1000}

# Pull GDP per capita
gdp_response = requests.get(f"https://api.worldbank.org/v2/country/{countries}/indicator/NY.GDP.PCAP.CD", params=params)
gdp_data = pd.DataFrame(gdp_response.json()[1])
gdp_data = gdp_data[['country', 'date', 'value']]
gdp_data['country'] = gdp_data['country'].apply(lambda x: x['value'])
gdp_data.columns = ['country', 'year', 'gdp_per_capita']
gdp_data['year'] = gdp_data['year'].astype(int)

# Pull debt-to-GDP
debt_response = requests.get(f"https://api.worldbank.org/v2/country/{countries}/indicator/GC.DOD.TOTL.GD.ZS", params={"format": "json", "per_page": 5000})
debt_data = pd.DataFrame(debt_response.json()[1])
debt_data = debt_data[['country', 'date', 'value']]
debt_data['country'] = debt_data['country'].apply(lambda x: x['value'])
debt_data.columns = ['country', 'year', 'debt_to_gdp']
debt_data['year'] = debt_data['year'].astype(int)

# Pull FDI
fdi_response = requests.get(f"https://api.worldbank.org/v2/country/{countries}/indicator/BX.KLT.DINV.WD.GD.ZS", params={"format": "json", "per_page": 5000})
fdi_data = pd.DataFrame(fdi_response.json()[1])
fdi_data = fdi_data[['country', 'date', 'value']]
fdi_data['country'] = fdi_data['country'].apply(lambda x: x['value'])
fdi_data.columns = ['country', 'year', 'fdi_pct_gdp']
fdi_data['year'] = fdi_data['year'].astype(int)

# Pull inflation
inf_response = requests.get(f"https://api.worldbank.org/v2/country/{countries}/indicator/FP.CPI.TOTL.ZG", params={"format": "json", "per_page": 5000})
inf_data = pd.DataFrame(inf_response.json()[1])
inf_data = inf_data[['country', 'date', 'value']]
inf_data['country'] = inf_data['country'].apply(lambda x: x['value'])
inf_data.columns = ['country', 'year', 'inflation']
inf_data['year'] = inf_data['year'].astype(int)

# Load into SQLite tables
gdp_data.to_sql('gdp', conn, index=False, if_exists='replace')
debt_data.to_sql('debt', conn, index=False, if_exists='replace')
fdi_data.to_sql('fdi', conn, index=False, if_exists='replace')
inf_data.to_sql('inflation', conn, index=False, if_exists='replace')

print("Tables loaded successfully")

Tables loaded successfully


In [2]:
# Q1: Overview of GDP data — checking structure and sample data
# HOW: Simple SELECT with LIMIT to preview the table
# WHY: Standard first step to understand what we're working with
pd.read_sql("SELECT * FROM gdp LIMIT 10", conn)

,country,year,gdp_per_capita
0,"Bahamas, The",2025,NaN
1,"Bahamas, The",2024,39455.446655
2,"Bahamas, The",2023,38231.774484
3,"Bahamas, The",2022,34957.161328
4,"Bahamas, The",2021,30367.860576
5,"Bahamas, The",2020,26178.753761
6,"Bahamas, The",2019,33640.336986
7,"Bahamas, The",2018,32642.335320
8,"Bahamas, The",2017,31875.488175
9,"Bahamas, The",2016,30616.536316


In [3]:
# Q2: Join GDP and debt tables to see fiscal health alongside economic output
# HOW: INNER JOIN on country and year so we only get rows where both indicators exist
# WHY: Investors need to see growth AND debt together to assess risk
pd.read_sql("""
    SELECT g.country, g.year, g.gdp_per_capita, d.debt_to_gdp
    FROM gdp g
    INNER JOIN debt d ON g.country = d.country AND g.year = d.year
    WHERE g.year >= 2000
    ORDER BY g.country, g.year
""", conn)

,country,year,gdp_per_capita,debt_to_gdp
0,"Bahamas, The",2000,24940.077509,18.752004
1,"Bahamas, The",2001,25371.923767,NaN
2,"Bahamas, The",2002,26781.619594,NaN
3,"Bahamas, The",2003,26429.124779,NaN
4,"Bahamas, The",2004,26650.293423,NaN
...,...,...,...,...
229,Trinidad and Tobago,2021,17712.567410,NaN
230,Trinidad and Tobago,2022,20750.520243,NaN
231,Trinidad and Tobago,2023,18308.453630,NaN
232,Trinidad and Tobago,2024,18733.411041,NaN


In [4]:
# Q3: Rank countries by average GDP per capita (2000+)
# HOW: Window function RANK() over average GDP per capita
# WHY: Shows where Jamaica sits relative to peers — important for investor context
pd.read_sql("""
    SELECT country, 
           ROUND(AVG(gdp_per_capita), 2) AS avg_gdp_per_capita,
           RANK() OVER (ORDER BY AVG(gdp_per_capita) DESC) AS gdp_rank
    FROM gdp
    WHERE year >= 2000
    GROUP BY country
""", conn)

,country,avg_gdp_per_capita,gdp_rank
0,"Bahamas, The",29739.94,1
1,Barbados,18583.21,2
2,Trinidad and Tobago,15877.64,3
3,Grenada,7849.13,4
4,Guyana,6468.54,5
5,Dominican Republic,6036.53,6
6,Belize,5802.75,7
7,Jamaica,5094.22,8
8,Haiti,1237.94,9


In [5]:
# Q4: Average GDP per capita by country with overall Caribbean average
# HOW: GROUP BY country with UNION ALL to add an overall average row (SQLite doesn't support ROLLUP, so we simulate it)
# WHY: Lets us compare each country to the regional average in one table
pd.read_sql("""
    SELECT country, ROUND(AVG(gdp_per_capita), 2) AS avg_gdp, COUNT(*) AS years_of_data
    FROM gdp
    WHERE year >= 2000
    GROUP BY country
    
    UNION ALL
    
    SELECT 'CARIBBEAN AVERAGE' AS country, ROUND(AVG(gdp_per_capita), 2), COUNT(*)
    FROM gdp
    WHERE year >= 2000
    
    ORDER BY avg_gdp DESC
""", conn)

,country,avg_gdp,years_of_data
0,"Bahamas, The",29739.94,26
1,Barbados,18583.21,26
2,Trinidad and Tobago,15877.64,26
3,CARIBBEAN AVERAGE,10743.32,234
4,Grenada,7849.13,26
5,Guyana,6468.54,26
6,Dominican Republic,6036.53,26
7,Belize,5802.75,26
8,Jamaica,5094.22,26
9,Haiti,1237.94,26


In [6]:
# Q5: Years where Jamaica's GDP growth exceeded the Caribbean average
# HOW: Subquery calculates the Caribbean average growth per year, outer query filters Jamaica
# WHY: Identifies Jamaica's outperformance periods — key for the investment case
pd.read_sql("""
    SELECT g1.year, 
           ROUND(g1.gdp_per_capita, 2) AS jamaica_gdp,
           ROUND(caribbean_avg.avg_gdp, 2) AS caribbean_avg_gdp
    FROM gdp g1
    INNER JOIN (
        SELECT year, AVG(gdp_per_capita) AS avg_gdp
        FROM gdp
        WHERE year >= 2000
        GROUP BY year
    ) caribbean_avg ON g1.year = caribbean_avg.year
    WHERE g1.country = 'Jamaica' AND g1.year >= 2000
    ORDER BY g1.year
""", conn)

,year,jamaica_gdp,caribbean_avg_gdp
0,2000,3453.09,6710.74
1,2001,3503.55,6814.11
2,2002,3680.65,7062.94
3,2003,3549.71,7202.62
4,2004,3807.56,7557.49
5,2005,4184.44,8521.66
6,2006,4417.00,9251.38
7,2007,4716.49,10053.30
8,2008,5029.80,10730.46
9,2009,4428.38,9509.16


In [7]:
# Q6: Year-over-year GDP per capita change for Jamaica using LAG
# HOW: LAG window function to get prior year's value, then calculate percentage change
# WHY: Shows Jamaica's growth trajectory year by year — the trend an investor needs to see
pd.read_sql("""
    SELECT year, 
           ROUND(gdp_per_capita, 2) AS gdp_per_capita,
           ROUND(LAG(gdp_per_capita) OVER (ORDER BY year), 2) AS prior_year_gdp,
           ROUND(((gdp_per_capita - LAG(gdp_per_capita) OVER (ORDER BY year)) 
                  / LAG(gdp_per_capita) OVER (ORDER BY year)) * 100, 2) AS yoy_growth_pct
    FROM gdp
    WHERE country = 'Jamaica' AND year >= 2000
    ORDER BY year
""", conn)

,year,gdp_per_capita,prior_year_gdp,yoy_growth_pct
0,2000,3453.09,NaN,NaN
1,2001,3503.55,3453.09,1.46
2,2002,3680.65,3503.55,5.05
3,2003,3549.71,3680.65,-3.56
4,2004,3807.56,3549.71,7.26
5,2005,4184.44,3807.56,9.90
6,2006,4417.00,4184.44,5.56
7,2007,4716.49,4417.00,6.78
8,2008,5029.80,4716.49,6.64
9,2009,4428.38,5029.80,-11.96


In [8]:
# Q7: Jamaica's full macro dashboard — joining all four tables
# HOW: Multiple LEFT JOINs to combine GDP, debt, FDI, and inflation for Jamaica
# WHY: Creates the complete picture an investor needs in one view
pd.read_sql("""
    SELECT g.year, 
           ROUND(g.gdp_per_capita, 2) AS gdp_per_capita,
           ROUND(d.debt_to_gdp, 2) AS debt_to_gdp,
           ROUND(f.fdi_pct_gdp, 2) AS fdi_pct_gdp,
           ROUND(i.inflation, 2) AS inflation
    FROM gdp g
    LEFT JOIN debt d ON g.country = d.country AND g.year = d.year
    LEFT JOIN fdi f ON g.country = f.country AND g.year = f.year
    LEFT JOIN inflation i ON g.country = i.country AND g.year = i.year
    WHERE g.country = 'Jamaica' AND g.year >= 2000
    ORDER BY g.year
""", conn)

,year,gdp_per_capita,debt_to_gdp,fdi_pct_gdp,inflation
0,2000,3453.09,98.33,4.66,8.17
1,2001,3503.55,117.54,6.27,6.80
2,2002,3680.65,127.78,4.57,7.08
3,2003,3549.71,127.43,7.25,10.09
4,2004,3807.56,122.01,5.50,13.55
5,2005,4184.44,121.00,5.67,15.05
6,2006,4417.00,117.70,7.06,8.56
7,2007,4716.49,112.99,6.34,9.24
8,2008,5029.80,120.34,10.04,22.02
9,2009,4428.38,134.68,4.00,9.59


In [9]:
# Q8: Decade-level comparison of GDP per capita by country
# HOW: Group by country and decade using CASE WHEN to bucket years
# WHY: Shows long-term trajectory — which countries are improving vs stagnating
pd.read_sql("""
    SELECT country,
           CASE 
               WHEN year BETWEEN 2000 AND 2009 THEN '2000s'
               WHEN year BETWEEN 2010 AND 2019 THEN '2010s'
               WHEN year >= 2020 THEN '2020s'
           END AS decade,
           ROUND(AVG(gdp_per_capita), 2) AS avg_gdp,
           COUNT(*) AS num_years
    FROM gdp
    WHERE year >= 2000
    GROUP BY country, decade
    ORDER BY country, decade
""", conn)

,country,decade,avg_gdp,num_years
0,"Bahamas, The",2000s,27491.74,10
1,"Bahamas, The",2010s,29939.00,10
2,"Bahamas, The",2020s,33838.20,6
3,Barbados,2000s,14096.70,10
4,Barbados,2010s,20682.34,10
5,Barbados,2020s,23357.98,6
6,Belize,2000s,5208.65,10
7,Belize,2010s,5942.65,10
8,Belize,2020s,6711.15,6
9,Dominican Republic,2000s,3624.85,10


In [10]:
# Q9: Countries with above-median FDI AND below-median inflation (attractive for investors)
# HOW: Two subqueries calculate the median thresholds, outer query filters countries that meet both
# WHY: Identifies which Caribbean economies are most investor-friendly on these two key metrics
pd.read_sql("""
    SELECT country, 
           ROUND(AVG(fdi_pct_gdp), 2) AS avg_fdi,
           ROUND(AVG(inflation), 2) AS avg_inflation
    FROM fdi f
    INNER JOIN inflation i USING (country, year)
    WHERE year >= 2010
    GROUP BY country
    HAVING AVG(fdi_pct_gdp) > (
        SELECT AVG(fdi_pct_gdp) FROM fdi WHERE year >= 2010
    )
    AND AVG(inflation) < (
        SELECT AVG(inflation) FROM inflation WHERE year >= 2010
    )
""", conn)

,country,avg_fdi,avg_inflation
0,Barbados,6.00,3.13
1,Grenada,12.70,1.21
2,Guyana,11.72,2.46


In [11]:
# Q10: Countries where debt-to-GDP decreased while GDP per capita increased (2015 vs 2020)
# HOW: Self-join on the GDP table for two time periods, joined with debt for both periods
# WHY: Identifies countries achieving the ideal combo — fiscal consolidation WITH growth
pd.read_sql("""
    SELECT g1.country,
           ROUND(g1.gdp_per_capita, 2) AS gdp_2015,
           ROUND(g2.gdp_per_capita, 2) AS gdp_2020,
           ROUND(d1.debt_to_gdp, 2) AS debt_2015,
           ROUND(d2.debt_to_gdp, 2) AS debt_2020,
           ROUND(g2.gdp_per_capita - g1.gdp_per_capita, 2) AS gdp_change,
           ROUND(d2.debt_to_gdp - d1.debt_to_gdp, 2) AS debt_change
    FROM gdp g1
    JOIN gdp g2 ON g1.country = g2.country
    JOIN debt d1 ON g1.country = d1.country AND g1.year = d1.year
    JOIN debt d2 ON g2.country = d2.country AND g2.year = d2.year
    WHERE g1.year = 2015 AND g2.year = 2020
    ORDER BY debt_change ASC
""", conn)

,country,gdp_2015,gdp_2020,debt_2015,debt_2020,gdp_change,debt_change
0,Belize,6154.62,5238.54,NaN,NaN,-916.08,NaN
1,Barbados,20424.21,19194.49,117.28,NaN,-1229.72,NaN
2,Dominican Republic,6800.95,7135.22,NaN,NaN,334.27,NaN
3,Grenada,8694.12,8968.56,NaN,NaN,274.43,NaN
4,Guyana,5640.42,6775.71,NaN,NaN,1135.29,NaN
5,Haiti,1411.13,1290.33,NaN,NaN,-120.80,NaN
6,Trinidad and Tobago,19887.23,15283.63,NaN,NaN,-4603.61,NaN
7,Jamaica,5339.31,5299.05,118.20,97.87,-40.26,-20.33
8,"Bahamas, The",30719.41,26178.75,47.62,79.04,-4540.65,31.42


In [12]:
# Q11: Running cumulative average of GDP per capita for Jamaica vs Caribbean overall
# HOW: Window function with ROWS BETWEEN to calculate expanding average over time
# WHY: Shows whether Jamaica's GDP is converging toward the Caribbean average over time
pd.read_sql("""
    SELECT year,
           ROUND(jamaica_gdp, 2) AS jamaica_gdp,
           ROUND(caribbean_avg_gdp, 2) AS caribbean_avg_gdp,
           ROUND(AVG(jamaica_gdp) OVER (ORDER BY year ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS jamaica_cumulative_avg,
           ROUND(AVG(caribbean_avg_gdp) OVER (ORDER BY year ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2) AS caribbean_cumulative_avg
    FROM (
        SELECT year,
               AVG(CASE WHEN country = 'Jamaica' THEN gdp_per_capita END) AS jamaica_gdp,
               AVG(gdp_per_capita) AS caribbean_avg_gdp
        FROM gdp
        WHERE year >= 2000
        GROUP BY year
    )
    ORDER BY year
""", conn)

,year,jamaica_gdp,caribbean_avg_gdp,jamaica_cumulative_avg,caribbean_cumulative_avg
0,2000,3453.09,6710.74,3453.09,6710.74
1,2001,3503.55,6814.11,3478.32,6762.42
2,2002,3680.65,7062.94,3545.76,6862.59
3,2003,3549.71,7202.62,3546.75,6947.60
4,2004,3807.56,7557.49,3598.91,7069.58
5,2005,4184.44,8521.66,3696.50,7311.59
6,2006,4417.00,9251.38,3799.43,7588.70
7,2007,4716.49,10053.30,3914.06,7896.78
8,2008,5029.80,10730.46,4038.03,8211.63
9,2009,4428.38,9509.16,4077.07,8341.38
